# 🌍 Seismic Fractal Analysis (SFA) - Google Colab Edition

## Catalog Precision-Dependent Fractal Saturation

**Resolución Multi-Escala mediante Inferencia Dimensional Bayesiana**

---

### 🎯 Tier-3 Open Science: Only Play!

✅ **Zero Installation** - Everything pre-installed in Colab  
✅ **One-Click Execution** - Execute cells sequentially  
✅ **Interactive Results** - Real-time interactive plots  
✅ **Educational Multinivel** - Researchers, Students, General Public

---

**Autores**: Facundo Firmenich¹'², Pau Firmenich¹, León Firmenich¹  
¹Centro de Estudios del Sur (CEDESUR), Argentina  
²Universitat de Barcelona, Spain

**Paper**: [JGR Solid Earth - Submitted]  
**Preprint**: https://eartharxiv.org/repository/view/11104/  
**GitHub**: https://github.com/FacundoFirmenich/SeismicFractalAnalysis  
**License**: GPLv3 Open Source

---

### 🚀 Quick Start

1. **Runtime** → Change runtime type → **GPU** (opcional, acelera 10-100×)
2. Ejecutar celdas **secuencialmente** (Ctrl+Enter o botón ▶️)
3. Explorar resultados interactivos
4. Modificar parámetros para experimentar

**Tiempo estimado**: 5-10 minutos (completo)


---

## 📦 1. Installation & Setup (Auto)

Automatic dependency installation. **Run only, do not modify.**


In [ ]:
%%capture
# Install SFA framework and dependencies
!pip install numpy scipy pandas matplotlib seaborn emcee obspy scikit-learn numba -q
!pip install plotly kaleido -q  # Interactive plots

# Clone SFA repository (or install from PyPI when published)
import os
if not os.path.exists('SeismicFractalAnalysis'):
    !git clone https://github.com/FacundoFirmenich/SeismicFractalAnalysis.git
    import sys
    sys.path.insert(0, '/content/SeismicFractalAnalysis')

print("✅ Installation complete!")

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
from datetime import datetime

# SFA Framework imports
try:
    from sfa.core import FractalDimensionEstimator
    from sfa.data import fetch_usgs_catalog
    from sfa.vis import plot_correlation_integral, plot_posterior_distribution
    print("✅ SFA Framework loaded successfully!")
except ImportError:
    print("⚠️ SFA not found. Using demo mode with synthetic data.")
    DEMO_MODE = True
else:
    DEMO_MODE = False

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print(f"📅 Execution timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---

## 📊 2. Interactive Demo: Fetch Real Earthquake Data

**Level**: 🎓 Student / 👨‍🔬 Researcher

Download seismic catalog **in real-time** from USGS.


In [ ]:
# User-configurable parameters
REGION_LAT = 35.0      # San Andreas Fault
REGION_LON = -120.0
RADIUS_KM = 192        # Search radius
MIN_MAGNITUDE = 2.5
START_DATE = "2010-01-01"
END_DATE = "2025-01-01"

print(f"🔍 Fetching earthquakes...")
print(f"   Region: ({REGION_LAT:.2f}, {REGION_LON:.2f})")
print(f"   Radius: {RADIUS_KM} km")
print(f"   Period: {START_DATE} → {END_DATE}")
print(f"   Min magnitude: M{MIN_MAGNITUDE}+")

if not DEMO_MODE:
    events_df = fetch_usgs_catalog(
        lat=REGION_LAT, lon=REGION_LON, radius_km=RADIUS_KM,
        start_time=START_DATE, end_time=END_DATE,
        min_magnitude=MIN_MAGNITUDE
    )
    
    print(f"\n✅ Downloaded {len(events_df)} events")
    print(f"   Magnitude range: M{events_df['mag'].min():.1f} - M{events_df['mag'].max():.1f}")
    print(f"   Depth range: {events_df['depth'].min():.1f} - {events_df['depth'].max():.1f} km")
    
    # Display sample
    display(events_df.head())
else:
    # Demo mode: generate synthetic data
    print("⚠️ DEMO MODE: Using synthetic fractal distribution")
    np.random.seed(42)
    N = 1000
    events_df = pd.DataFrame({
        'latitude': REGION_LAT + np.random.randn(N) * 2,
        'longitude': REGION_LON + np.random.randn(N) * 2,
        'depth': np.abs(np.random.exponential(10, N)),
        'mag': 2.5 + np.random.exponential(1, N)
    })
    print(f"✅ Generated {len(events_df)} synthetic events")

---

## 📈 3. Interactive Visualization: 3D Seismicity Map

**Level**: 🌍 General Public / 🎓 Student

Interactive 3D visualization of spatial seismic distribution.


In [ ]:
# Create interactive 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=events_df['longitude'],
    y=events_df['latitude'],
    z=events_df['depth'],
    mode='markers',
    marker=dict(
        size=events_df['mag'] * 2,
        color=events_df['mag'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Magnitude"),
        line=dict(width=0.5, color='white')
    ),
    text=[f"M{m:.1f}, Depth {d:.1f}km" for m, d in zip(events_df['mag'], events_df['depth'])],
    hoverinfo='text'
)])

fig.update_layout(
    title=f"Seismic Distribution: {len(events_df)} Events (2010-2025)",
    scene=dict(
        xaxis_title='Longitude (°)',
        yaxis_title='Latitude (°)',
        zaxis_title='Depth (km)',
        zaxis=dict(autorange='reversed')  # Depth increases downward
    ),
    width=900,
    height=700
)

fig.show()

print("💡 Tip: Rotate con mouse, zoom con scroll, pan con Shift+drag")

---

## 🧮 4. Fractal Dimension Analysis (Grassberger-Procaccia)

**Level**: 👨‍🔬 Researcher

Correlation dimension D₂ calculation using Grassberger-Procaccia algorithm.


In [ ]:
# Extract coordinates
coords = events_df[['latitude', 'longitude', 'depth']].values

print(f"📐 Computing correlation dimension D₂...")
print(f"   N events: {len(coords)}")
print(f"   Method: Grassberger-Procaccia (1983)")
print(f"   Slope estimator: Theil-Sen (robust)")

if not DEMO_MODE:
    estimator = FractalDimensionEstimator(
        n_bootstrap=100,
        knn_sparse=True,  # Memory efficient for large N
        random_state=42
    )
    
    d2, d2_sem, diagnostics = estimator.compute_gp_dimension(
        coords, return_diagnostics=True
    )
    
    print(f"\n✅ D₂ = {d2:.3f} ± {d2_sem:.3f}")
    print(f"   95% CI: [{d2 - 1.96*d2_sem:.3f}, {d2 + 1.96*d2_sem:.3f}]")
    print(f"   R² linearity: {diagnostics['r_squared']:.4f}")
    
    # Plot correlation integral
    plot_correlation_integral(diagnostics, save_path=None)
else:
    # Demo: simulate D₂ calculation
    d2 = 2.15 + np.random.randn() * 0.05
    d2_sem = 0.03
    print(f"\n✅ D₂ = {d2:.3f} ± {d2_sem:.3f} (DEMO)")
    print("   [Demo mode: Real calculation requires SFA framework]")

---

## 🎲 5. Bayesian D₃ Inference (MCMC Sampling)

**Level**: 👨‍🔬 Researcher Avanzado

Bayesian inference of volumetric dimension D₃ using MCMC (emcee).


In [ ]:
print("🎲 Bayesian MCMC Inference...")
print("   Prior: Beta(7.5, 2.5) → mode ≈ 2.9")
print("   Walkers: 32")
print("   Steps: 5000")
print("   Sampler: emcee (Foreman-Mackey 2013)")

if not DEMO_MODE:
    posterior_samples, diagnostics = estimator.bayesian_d3_inference(
        coords,
        n_walkers=32,
        n_steps=5000,
        prior_params=(7.5, 2.5),
        return_diagnostics=True
    )
    
    d3_mean = np.mean(posterior_samples)
    d3_std = np.std(posterior_samples)
    d3_median = np.median(posterior_samples)
    kl_divergence = diagnostics['kl_divergence']
    posterior_mass = diagnostics['posterior_concentration_2.98_3.00']
    
    print(f"\n✅ D₃ posterior:")
    print(f"   Mean: {d3_mean:.3f} ± {d3_std:.3f}")
    print(f"   Median: {d3_median:.3f}")
    print(f"   KL divergence: {kl_divergence:.2f} nats")
    print(f"   P(D₃ ∈ [2.98,3.00]): {posterior_mass:.2%}")
    
    # Triple validation decision
    print("\n🔍 Triple Validation:")
    print(f"   ✓ KL > 2.0: {'PASS' if kl_divergence > 2.0 else 'FAIL'}")
    print(f"   ✓ Posterior @ boundary < 5%: {'PASS' if posterior_mass < 0.05 else 'FAIL'}")
    print(f"   ✓ Zaccagnino S > 0.95: [Computing...]")
    
    # Plot posterior distribution
    plot_posterior_distribution(posterior_samples, save_path=None)
else:
    print("\n⚠️ DEMO MODE: Bayesian inference requires full SFA framework")
    print("   Install SFA to run real Bayesian MCMC analysis")

---

## 🎯 6. Interpretation Guide (Multinivel)

### 🌍 General Public

**D₂ ≈ 2.15**: Earthquakes in this region are organized in quasi-planar structures (like cracks in multiple layers). They are NOT distributed completely random in 3D.

**¿Qué significa?** Las fallas geológicas controlan DÓNDE ocurren los terremotos. Conocer esto ayuda a predecir zonas de riesgo.

---

### 🎓 Student

**D₂ = 2.15 ± 0.03**:
- D₂ < 3.0 → NOT a pure volumetric fractal structure
- D₂ ≈ 2.0 → planar tendency (faults dominate)
- Incertidumbre ±0.03 → statistically robust (bootstrap N=100)

**Bayesian D₃**:
- Si P(D₃ ∈ [2.98,3.00]) > 10% → SATURADO (artefacto precisión catálogo)
- Si < 5% → DATA-DRIVEN (dimensión genuina)

---

### 👨‍🔬 Researcher

**Hallazgo clave**: Precision catalog dependency
- USGS (σ>5km) → D₃=3.00 saturated (prior-dominated)
- Hi-Net (σ<2km) → D₃=2.82-2.94 data-driven
- Threshold: σ ≈ 2-3 km

**Triple Validation**:
1. KL divergence > 2.0 nats
2. Posterior @ [2.98,3.00] < 5%
3. Zaccagnino Mmin-independence S > 0.95

**Implicación**: Hi-Net precision = gold standard for definitive dimensional claims.


---

## 📚 7. References & Further Reading

### Core Papers

1. **Grassberger & Procaccia (1983)**: *Measuring the strangeness of strange attractors*. Physica D. [Original GP algorithm]
2. **Firmenich et al. (2025)**: *Catalog Precision-Dependent Fractal Saturation*. JGR Solid Earth. [This work]
3. **Zaccagnino et al. (2022, 2023)**: *Scaling behavior of seismicity*. PEPI. [Mmin-independence stability]

### Software

- **SFA GitHub**: https://github.com/FacundoFirmenich/SeismicFractalAnalysis
- **USGS API**: https://earthquake.usgs.gov/fdsnws/event/1/
- **emcee MCMC**: https://emcee.readthedocs.io/

### Data Sources

- **USGS**: Pan-American catalog (σ~5-10 km)
- **Hi-Net**: Japanese precision network (σ<2 km) - https://doi.org/10.17598/NIED.0003
- **ISC-GEM**: Global M8+ catalog (σ≈2-3 km) - https://doi.org/10.31905/D808B825

---

## 📜 License & Citation

**License**: GPLv3 Open Source  
**Citation**:

```bibtex
@article{Firmenich2025_SFA,
  title={Catalog Precision-Dependent Fractal Saturation in Seismic Volumetry},
  author={Firmenich, Facundo and Firmenich, Pau and Firmenich, León},
  journal={Journal of Geophysical Research: Solid Earth},
  year={2025},
  note={Submitted}
}
```

---

## 💻 Hardware Used

**Democratización Tier-3**: Esta investigación tier-1 mundial (target JGR/Nature) fue ejecutada en:

- **Laptop**: HP Pavilion 15 (2016) - Mid-range
- **CPU**: Intel Core i5-6200U @ 2.3GHz
- **RAM**: 8 GB DDR4
- **GPU**: Intel HD Graphics 520 (integrated)
- **Original cost**: ~$500 USD

**Mensaje**: Excelencia científica NO requiere supercomputadoras inaccesibles. Ciencia para TODOS.


---

## 🎓 Next Steps

### Experiment

1. Modify parameters in cell #2 (región, radio, fechas)
2. Execute analysis for YOUR region of interest
3. Compare D₂ across different tectonic zones

### Learn More

- **Tutorial completo**: `notebooks/Tutorial_SFA_QuickStart.ipynb`
- **Mathematical appendix**: `docs/MATHEMATICAL_APPENDIX.md`
- **API docs**: https://facundofirmenich.github.io/SeismicFractalAnalysis/

### Contribute

- **GitHub Issues**: Report bugs or suggest features
- **Pull Requests**: Contribute código (ver CONTRIBUTING.md)
- **Traducciones**: Ayudar con docs ES/PT/EN

---

**Questions?** f.firmenich@cedesur.org

**Timestamp**: `2025-12-14 14:20:00 UTC`
